In [5]:
"""
Ford–Fulkerson repeatedly:

Finds a path from source → sink
Pushes as much flow as possible through it (the bottleneck)
Updates the network to reflect used capacity

Think of it like pushing water through pipes until no more can reach the sink.

Ford–Fulkerson is basically:
repeat:
    find path p
    send flow f_p

max flow = no augmenting paths exist
"""

'\nFord–Fulkerson repeatedly:\n\nFinds a path from source → sink\nPushes as much flow as possible through it (the bottleneck)\nUpdates the network to reflect used capacity\n\nThink of it like pushing water through pipes until no more can reach the sink.\n\nFord–Fulkerson is basically:\nrepeat:\n    find path p\n    send flow f_p\n\nmax flow = no augmenting paths exist\n'

In [2]:
from collections import deque

def bfs(residual, s, t, parent):
    visited = set()
    queue = deque([s])
    visited.add(s)

    while queue:
        u = queue.popleft()
        for v in residual[u]:
            if v not in visited and residual[u][v] > 0:
                visited.add(v)
                parent[v] = u
                if v == t:
                    return True
                queue.append(v)
    return False


def ford_fulkerson(G, s, t):
    residual = {u: {} for u in G}
    for u in G:
        for v in G[u]:
            residual[u][v] = G[u][v]
            if v not in residual:
                residual[v] = {}
            if u not in residual[v]:
                residual[v][u] = 0  # reverse edge

    parent = {}
    max_flow = 0

    while bfs(residual, s, t, parent):
        path_flow = float('inf')
        v = t
        while v != s:
            u = parent[v]
            path_flow = min(path_flow, residual[u][v])
            v = u

        v = t
        while v != s:
            u = parent[v]
            residual[u][v] -= path_flow
            residual[v][u] += path_flow
            v = u

        max_flow += path_flow

    return max_flow

In [3]:
G = {
    's': {'a': 3, 'b': 2},
    'a': {'b': 1, 't': 2},
    'b': {'t': 3},
    't': {}
}

s = 's'
t = 't'

max_flow = ford_fulkerson(G, s, t)
print(max_flow)

5


In [6]:
"""
Capacity = fixed limit
Flow = what you've used
Residual graph = what you can still change

Capacity = 12 liters/sec pipe
Flow = 12 liters/sec already flowing
Residual forward = 0 (no space left)
Residual backward = 12 (you can push water back)

Disallowing edges into s in the residual network:
does not remove any necessary augmenting paths
does not affect reachability of t
preserves the cut-based optimality argument

| Property             | Holds for (f ^ f')? |
| -------------------- | ------------------- |
| Flow conservation    | Yes                 |
| Capacity constraints | No                  |
Flow conservation is linear, so adding flows preserves it.
Capacity constraints are upper bounds, so adding flows can overflow edges.

Edge connectivity asks:
What is the fewest number of edges whose removal can isolate part of the graph?

Max-flow answers exactly this:
How many edge-disjoint routes can survive between two vertices?

The weakest pair determines the graph's overall edge connectivity.

A positive-flow edge into the source means:
Some flow left the source earlier and came back around.

That is pure circulation (a cycle), not useful source-to-sink flow.
So we can safely remove that circulation without changing the total flow value.

In max-flow problems, a cut answers:
What is the smallest set of connections whose removal stops flow from reaching the destination?

Max-Flow Min-Cut Theorem
maximum possible flow = minimum cut capacity

cuts are usually:
analytical,
not operational.

The minimum cut tells you:
If you want MORE total flow, these are the edges you must upgrade.

In practice:
cuts identify weak points.

A minimum cut does not remove the bottleneck to allow more flow; rather,
it identifies the bottleneck whose total capacity limits the maximum possible flow through the network.
"""

"\nCapacity = fixed limit\nFlow = what you've used\nResidual graph = what you can still change\n\nCapacity = 12 liters/sec pipe\nFlow = 12 liters/sec already flowing\nResidual forward = 0 (no space left)\nResidual backward = 12 (you can push water back)\n\nDisallowing edges into s in the residual network:\ndoes not remove any necessary augmenting paths\ndoes not affect reachability of t\npreserves the cut-based optimality argument\n\n| Property             | Holds for (f ^ f')? |\n| -------------------- | ------------------- |\n| Flow conservation    | Yes                 |\n| Capacity constraints | No                  |\nFlow conservation is linear, so adding flows preserves it.\nCapacity constraints are upper bounds, so adding flows can overflow edges.\n\nEdge connectivity asks:\nWhat is the fewest number of edges whose removal can isolate part of the graph?\n\nMax-flow answers exactly this:\nHow many edge-disjoint routes can survive between two vertices?\n\nThe weakest pair determin

In [8]:
"""
Ignore all "small pipes"
Only use "large pipes"

Then gradually lower the threshold:

8 → 4 → 2 → 1

until every possible augmenting path is considered.
"""
from collections import deque
import math

class Edge:
    def __init__(self, to, capacity):
        self.to = to
        self.capacity = capacity
        self.rev = None


class MaxFlowScaling:
    def __init__(self, n):
        self.n = n
        self.graph = [[] for _ in range(n)]

    def add_edge(self, u, v, capacity):
        forward = Edge(v, capacity)
        backward = Edge(u, 0)

        forward.rev = backward
        backward.rev = forward

        self.graph[u].append(forward)
        self.graph[v].append(backward)

    def bfs(self, s, t, K):
        parent = [None] * self.n
        visited = [False] * self.n

        q = deque([s])
        visited[s] = True

        while q:
            u = q.popleft()

            for edge in self.graph[u]:
                if (not visited[edge.to]
                        and edge.capacity >= K):

                    visited[edge.to] = True
                    parent[edge.to] = (u, edge)

                    if edge.to == t:
                        return parent

                    q.append(edge.to)

        return None

    def max_flow_by_scaling(self, s, t):
        C = 0
        for u in range(self.n):
            for edge in self.graph[u]:
                C = max(C, edge.capacity)

        """
        Example: 
        C = 13
        Binary:
        1101 needs 4 bits.
        13.bit_length() == 4
        C.bit_length() - 1 = 4 - 1 = 3
        = floor(log_2*C)
        because: 2^3 <= 13 < 2^4

        1 << 0 = 1
        1 << 1 = 2
        1 << 2 = 4
        1 << 3 = 8
        1 << 4 = 16

        K = 1 << 3 = 8
        8 = 2^floor(log_2*13)
        """
        # Largest power of 2 <= C 
        K = 1 << (C.bit_length() - 1) # equivlent: K = 2 ** int(math.log2(C)) but bitwise is faster
        flow = 0

        while K >= 1:
            while True:
                parent = self.bfs(s, t, K)

                if parent is None:
                    break

                bottleneck = float('inf')
                v = t

                while v != s:
                    u, edge = parent[v]
                    bottleneck = min(bottleneck, edge.capacity)
                    v = u

                # Augment flow
                v = t
                while v != s:
                    u, edge = parent[v]

                    edge.capacity -= bottleneck
                    edge.rev.capacity += bottleneck

                    v = u

                flow += bottleneck

            K //= 2

        return flow

In [9]:
mf = MaxFlowScaling(4)

mf.add_edge(0, 1, 10)
mf.add_edge(0, 2, 5)
mf.add_edge(1, 2, 15)
mf.add_edge(1, 3, 10)
mf.add_edge(2, 3, 10)

print(mf.max_flow_by_scaling(0, 3))

15


In [10]:
# Example Network
#
#        8
#   s ------> a
#   |          \
# 5 |           \ 3
#   |            \
#   v             v
#   b ----------> t
#         4
#
# Capacities:
# s -> a = 8
# s -> b = 5
# a -> t = 3
# b -> t = 4


# -----------------------------------
# STEP 1: Find maximum edge capacity
# -----------------------------------

C = 8

# Largest power of 2 <= 8
K = 8

# ===================================
# SCALING PHASE K = 8
# ===================================

# Only edges with residual capacity >= 8 allowed

# Allowed edges:
# s -> a = 8

# But:
# a -> t = 3   (too small)

# Therefore:
# No augmenting path exists

# Halve K
K = 4

# ===================================
# SCALING PHASE K = 4
# ===================================

# Allowed edges:
# s -> a = 8
# s -> b = 5
# b -> t = 4

# Edge:
# a -> t = 3
# still excluded

# Found augmenting path:
# s -> b -> t

# Bottleneck:
min(5, 4) = 4

# Push 4 units of flow

# Residual capacities now:
# s -> b = 1
# b -> t = 0

# Current total flow = 4

# Try another K=4 augmenting path

# Remaining usable edges:
# s -> a = 8

# But:
# a -> t = 3  (still too small)

# No augmenting path exists


# Halve K
K = 2

# ===================================
# SCALING PHASE K = 2
# ===================================

# Allowed edges:
# s -> a = 8
# a -> t = 3

# Found augmenting path:
# s -> a -> t

# Bottleneck:
min(8, 3) = 3

# Push 3 units of flow

# Residual capacities now:
# s -> a = 5
# a -> t = 0

# Current total flow = 7

# Try another K=2 augmenting path, No path exists

# Halve K
K = 1

# ===================================
# SCALING PHASE K = 1
# ===================================

# Remaining forward capacities:
# s -> a = 5
# s -> b = 1

# But:
# a -> t = 0
# b -> t = 0

# No augmenting path exists, Algorithm terminates

# ===================================
# FINAL RESULT
# ===================================

# Maximum flow = 7

SyntaxError: cannot assign to function call (3518879467.py, line 63)

| Algorithm                        | What improves per step            | Behavior             |
| -------------------------------- | --------------------------------- | -------------------- |
| Ford–Fulkerson (arbitrary paths) | +1 unit                           | linear / can be slow |
| Edmonds–Karp                     | shortest path structure           | polynomial           |
| Widest-path version              | removes fraction of remaining gap | exponential decay    |


In [11]:
"""
Even though it improves Ford–Fulkerson, it is still not the best known.
Better algorithms exist:

| Algorithm                       | Complexity                                      |
| ------------------------------- | ----------------------------------------------- |
| Dinic                           | (O(EV^{2/3})) or (O(E\sqrt{V})) (unit networks) |
| Push-relabel                    | very fast in practice                           |
| Scaling + blocking flow hybrids | even better                                     |

Widest-path greedy choice is:
"maximize immediate progress"

max-flow is a global structure problem

greedy local "best bottleneck" is not always aligned with best global blocking-flow structure
modern algorithms focus on layering / blocking flows, not single-path greediness

fewer augmentations ≠ fastest algorithm
cost per augmentation matters more
structure-based algorithms (Dinic, push-relabel) outperform both EK and widest-path scaling

Widest-path FF is like:
taking the biggest shovel to dig fewer holes

But modern max-flow is:
redesigning the entire digging strategy so you barely need holes at all
"""

'\nEven though it improves Ford–Fulkerson, it is still not the best known.\nBetter algorithms exist:\n\n| Algorithm                       | Complexity                                      |\n| ------------------------------- | ----------------------------------------------- |\n| Dinic                           | (O(EV^{2/3})) or (O(E\\sqrt{V})) (unit networks) |\n| Push-relabel                    | very fast in practice                           |\n| Scaling + blocking flow hybrids | even better                                     |\n\nWidest-path greedy choice is:\n"maximize immediate progress"\n\nmax-flow is a global structure problem\n\ngreedy local "best bottleneck" is not always aligned with best global blocking-flow structure\nmodern algorithms focus on layering / blocking flows, not single-path greediness\n\nfewer augmentations ≠ fastest algorithm\ncost per augmentation matters more\nstructure-based algorithms (Dinic, push-relabel) outperform both EK and widest-path scaling\n\nW

In [12]:
# ============================================================
# WIDEST PATH ANALYSIS
# Showing how Δ_i shrinks in widest-augmenting-path Ford-Fulkerson
# ============================================================

# ------------------------------------------------------------
# SETUP (conceptual example)
# ------------------------------------------------------------

# Let:
# |f*| = 100   (maximum flow value)
# |E|  = 5     (number of edges)

# So the key bound becomes:
# Δ_{i+1} <= Δ_i * (1 - 1/|E|) = Δ_i * (4/5)

# ------------------------------------------------------------
# INITIAL GAP
# ------------------------------------------------------------

Delta = 100
E = 5

print("i = 0, Delta =", Delta)

# ------------------------------------------------------------
# ITERATIVE SHRINKAGE (widest path effect)
# ------------------------------------------------------------

for i in range(1, 11):

    # each augmentation removes at least a 1/E fraction
    Delta = Delta * (1 - 1/E)

    print(f"i = {i}, Delta ≈ {Delta:.4f}")

# ------------------------------------------------------------
# WHAT THIS IS SHOWING
# ------------------------------------------------------------

# Each step:
# Δ_i → Δ_i * (1 - 1/|E|)

# For |E| = 5:
# factor = 0.8

# So sequence is:
#
# 100
# 80
# 64
# 51.2
# 40.96
# 32.77
# 26.21
# 20.97
# 16.78
# 13.42
# 10.74
#
# (exponential decay)

# ------------------------------------------------------------
# CONNECTION TO THEORY
# ------------------------------------z------------------------

# General form:
# Δ_i <= |f*| * (1 - 1/|E|)^i
#
# Using inequality:
# (1 - 1/|E|)^i <= e^{-i/|E|}
#
# So:
# Δ_i < |f*| * e^{-i/|E|}
#
# ------------------------------------------------------------
# STOP CONDITION
# ------------------------------------------------------------

# We stop when Δ_i < 1

# Solve:
# |f*| * e^{-i/|E|} < 1

# => e^{-i/|E|} < 1/|f*|
# => i > |E| * ln(|f*|)

print("\nStop condition reached when i > |E| ln(|f*|)")
print("Here: i > 5 * ln(100) ≈", 5 * 4.605)

i = 0, Delta = 100
i = 1, Delta ≈ 80.0000
i = 2, Delta ≈ 64.0000
i = 3, Delta ≈ 51.2000
i = 4, Delta ≈ 40.9600
i = 5, Delta ≈ 32.7680
i = 6, Delta ≈ 26.2144
i = 7, Delta ≈ 20.9715
i = 8, Delta ≈ 16.7772
i = 9, Delta ≈ 13.4218
i = 10, Delta ≈ 10.7374

Stop condition reached when i > |E| ln(|f*|)
Here: i > 5 * ln(100) ≈ 23.025000000000002


In [ ]:
"""
A cut is a partition of vertices.

When you contract:
you are reducing the number of vertices
so you are simplifying the graph
but trying not to destroy the structure of the minimum cut

u and v become a single "supernode"
everything else stays the same
edges just get rewired to the supernode

lemma behind Karger's contraction algorithm:
if you randomly contract edges
you rarely contract a min-cut edge
so with good probability, you preserve the true min cut
"""